# 📊 Analyse Exploratoire des Données (EDA) : Prédiction de l'Attrition des Employés

**Contexte :** Dataset *IBM HR Analytics Employee Attrition & Performance* (1 470 collaborateurs, 35 variables).  
**Problématique métier :** Le départ d'un collaborateur (*Employee Attrition*) engendre des coûts directs de recrutement et de formation, ainsi qu'une perte de compétences stratégiques. Comprendre les signaux précurseurs du turnover est un enjeu clé pour la direction des ressources humaines afin de cibler des actions de fidélisation efficaces.

---

### 🎯 Objectifs de l'Analyse Exploratoire
1. **Inspection & Nettoyage :** Vérifier l'exhaustivité des données, éliminer les variables sans valeur informationnelle (constantes et identifiants techniques) et classifier correctement les types réels (distinction entre mesures quantitatives continues et évaluations ordinales).
2. **Analyse Univariée :** Dresser le portrait d'ensemble des collaborateurs, caractériser la variable cible `Attrition` (mesurer le déséquilibre de classe) et étudier les distributions des variables explicatives (asymétries, dispersion, outliers).
3. **Analyse Bivariée :** Identifier les facteurs directement associés au départ en comparant systématiquement les employés qui restent (`Attrition = No`) et ceux qui partent (`Attrition = Yes`).
4. **Analyse Multivariée :** Détecter les fortes corrélations et la multicolinéarité pour préparer efficacement l'étape de modélisation prédictive.


## 1. Chargement des Bibliothèques et Première Inspection des Données


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration esthétique globale des graphiques
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.autolayout"] = True

# Chargement du jeu de données brut
df = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Aperçu des 5 premières lignes
df.head()


In [ ]:
# Dimensions du dataset (nombre de lignes et de colonnes)
print(f"Nombre d'employés (observations) : {df.shape[0]}")
print(f"Nombre de variables (colonnes)    : {df.shape[1]}")


In [ ]:
# Structure technique globale des variables et types de données
df.info()


In [ ]:
# Vue d'ensemble statistique brute (numériques et textuelles)
df.describe(include="all")


In [ ]:
# Vérification de l'exhaustivité des données (détection des NaN)
missing_total = df.isna().sum().sum()
print(f"Nombre total de valeurs manquantes : {missing_total}")


### 🔍 Constats sur l'état initial des données
- **Exhaustivité :** Aucune valeur manquante (`0 NaN`). Le dataset est complet et prêt à être analysé sans imputation préalable.
- **Volumétrie :** 1 470 observations et 35 variables.
- **Remarque sur les types techniques :** Le fichier contient 26 colonnes de type entier (`int64`) et 9 colonnes de type texte (`object`). Cependant, comme souligné dans le document de cadrage, plusieurs variables codées par des entiers représentent en réalité des **échelles ordinales** (notes de 1 à 4 ou 1 à 5).


## 2. Classification Métier des Variables & Nettoyage

Une étape essentielle de préparation consiste à séparer les variables selon leur **nature métier** réelle :
1. **Variables à exclure :** 
   - `EmployeeNumber` : simple identifiant technique unique sans pouvoir prédictif.
   - `EmployeeCount` (constante = 1), `StandardHours` (constante = 80), `Over18` (constante = 'Y') : n'apportent aucune variance.
2. **Variable Cible :** `Attrition` (binaire : `Yes` / `No`).
3. **Variables Catégorielles Ordinales :** Niveaux d'éducation, notes de satisfaction, implication, niveaux hiérarchiques et stock-options (codées numériquement mais ordonnées).
4. **Variables Catégorielles Nominales :** Postes, départements, situation matrimoniale, etc.
5. **Variables Numériques Réelles :** Mesures quantitatives réelles (salaires, âges, distances, ancienneté).


In [ ]:
# 1. Variables à exclure (constantes et identifiant technique)
excluded_columns = [
    "EmployeeNumber",
    "EmployeeCount",
    "StandardHours",
    "Over18"
]

# 2. Variable cible (problème de classification binaire)
target_col = "Attrition"

# 3. Variables ordinales (notes d'évaluation et paliers hiérarchiques)
ordinal_columns = [
    "Education",
    "EnvironmentSatisfaction",
    "JobInvolvement",
    "JobSatisfaction",
    "JobLevel",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "WorkLifeBalance",
    "StockOptionLevel"
]

# 4. Variables nominales (catégories qualitatives non ordonnées)
nominal_columns = [
    col
    for col in df.select_dtypes(include=["object", "string"]).columns
    if col != target_col and col not in excluded_columns
]

# 5. Véritables variables numériques (mesures quantitatives continues et comptages)
numeric_columns = [
    col
    for col in df.select_dtypes(include="number").columns
    if col not in ordinal_columns and col not in excluded_columns
]

# Nettoyage : suppression effective des variables exclues
df = df.drop(columns=excluded_columns, errors='ignore')

# Bilan de la classification
print(f"Variable cible : {target_col}")
print(f"Variables nominales ({len(nominal_columns)}) : {nominal_columns}")
print(f"Variables ordinales ({len(ordinal_columns)}) : {ordinal_columns}")
print(f"Variables numériques ({len(numeric_columns)}) : {numeric_columns}")
print(f"Total des variables exploitables : {df.shape[1]} (dont 1 cible)")


In [ ]:
# Tableau récapitulatif du profil des 31 variables exploitables
profile = pd.DataFrame({
    "Type technique": df.dtypes,
    "Valeurs uniques": df.nunique(),
    "Valeurs manquantes": df.isna().sum(),
    "Taux de complétude (%)": (1 - df.isna().mean()) * 100
})

display(profile.sort_values(by="Valeurs uniques"))


## 3. Analyse Univariée

L'analyse univariée examine chaque variable isolément afin d'en comprendre la distribution, la dispersion et d'identifier les anomalies éventuelles.

---

### 3.1. Analyse de la Variable Cible : `Attrition`
L'analyse de la variable cible est primordiale pour diagnostiquer le **degré de déséquilibre de classe** (*class imbalance*).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = df["Attrition"].value_counts()
pcts = df["Attrition"].value_counts(normalize=True) * 100

# Barplot avec annotations des effectifs et proportions
sns.barplot(
    x=counts.index,
    y=counts.values,
    hue=counts.index,
    palette=["#4C72B0", "#C44E52"],
    legend=False,
    ax=axes[0]
)
axes[0].set_title("Répartition de l'Attrition (Effectifs et %)", fontweight="bold")
axes[0].set_ylabel("Nombre d'employés")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 25, f"{v} ({pcts.iloc[i]:.1f}%)", ha="center", fontweight="bold")
axes[0].set_ylim(0, max(counts.values) * 1.15)

# Donut chart
axes[1].pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    colors=["#4C72B0", "#C44E52"],
    startangle=90,
    wedgeprops=dict(width=0.4)
)
axes[1].set_title("Proportion de la Cible", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Taux d'attrition de référence : {pcts['Yes']:.1f}% (237 départs sur 1 470 collaborateurs).")


### 💡 Constat Clé sur la Variable Cible
- **Déséquilibre de classe marqué :** Seuls **16.1%** des collaborateurs ont quitté l'entreprise (237 départs vs 1 233 employés restants).
- **Conséquence pour le Machine Learning :** L'évaluation par la simple *Accuracy* (exactitude globale) sera fortement trompeuse : un modèle prédisant systématiquement `No` obtiendrait artificiellement 83.9% de précision sans jamais détecter un seul départ. Pour évaluer nos futurs modèles, il faudra privilégier :
  - Le **Recall** (taux de détection des vrais départs `Yes`),
  - Le **F1-Score** (compromis précision-rappel),
  - La courbe **Precision-Recall AUC (PR-AUC)**.


---

### 3.2. Analyse des Variables Numériques Réelles
Nous analysons les 14 variables numériques continues et discrètes afin de mesurer leur tendance centrale (moyenne, médiane), leur étalement (écart-type, IQR), leur asymétrie (*skewness*) et la présence de valeurs extrêmes (*outliers*).


In [ ]:
# Tableau statistique complet des 14 variables numériques
stats_list = []
for col in numeric_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    pct_outliers = (n_outliers / len(df)) * 100
    
    stats_list.append({
        "Variable": col,
        "Moyenne": round(df[col].mean(), 2),
        "Écart-type": round(df[col].std(), 2),
        "Médiane": round(df[col].median(), 2),
        "IQR": round(iqr, 2),
        "Min": df[col].min(),
        "Max": df[col].max(),
        "Skewness": round(df[col].skew(), 2),
        "Outliers (n)": n_outliers,
        "Outliers (%)": f"{pct_outliers:.1f}%"
    })

stats_df = pd.DataFrame(stats_list)
display(stats_df.sort_values(by="Skewness", ascending=False))


In [ ]:
# Vue d'ensemble de la distribution des 14 variables numériques
# Ligne rouge pointillée = Moyenne | Ligne verte pleine = Médiane
fig, axes = plt.subplots(5, 3, figsize=(15, 15))
axes = axes.flatten()

for i, col in enumerate(numeric_columns):
    ax = axes[i]
    sns.histplot(df[col], kde=True, ax=ax, color="#1f77b4", bins=20)
    ax.axvline(df[col].mean(), color="red", linestyle="--", linewidth=1.2, label=f"Moy: {df[col].mean():.1f}")
    ax.axvline(df[col].median(), color="green", linestyle="-", linewidth=1.2, label=f"Méd: {df[col].median():.1f}")
    ax.set_title(f"{col} (skew={df[col].skew():.2f})", fontsize=10, fontweight="bold")
    ax.set_ylabel("")
    ax.legend(fontsize=8)

# Masquer le 15ème subplot vide
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
def plot_num_detail(df, col):
    """Affiche un boxplot et un histogramme avec KDE alignés pour étudier finement la dispersion et les outliers d'une variable."""
    fig, (ax_box, ax_hist) = plt.subplots(
        2, 1, sharex=True, gridspec_kw={"height_ratios": (0.2, 0.8)}, figsize=(9, 4)
    )
    sns.boxplot(x=df[col], ax=ax_box, color="#9ecae1")
    sns.histplot(df[col], kde=True, ax=ax_hist, color="#3182bd")
    ax_box.set(xlabel="")
    ax_hist.axvline(df[col].mean(), color="red", linestyle="--", label=f"Moyenne ({df[col].mean():.1f})")
    ax_hist.axvline(df[col].median(), color="green", linestyle="-", label=f"Médiane ({df[col].median():.1f})")
    ax_hist.set_title(f"Distribution détaillée et Outliers : {col} (Skewness={df[col].skew():.2f})", fontweight="bold", fontsize=11)
    ax_hist.legend()
    plt.tight_layout()
    plt.show()

# Inspection détaillée des deux variables les plus asymétriques
plot_num_detail(df, "MonthlyIncome")
plot_num_detail(df, "YearsAtCompany")


### 💡 Synthèse & Enseignements de l'Analyse Univariée (Variables Numériques)

1. **Variables asymétriques à droite avec outliers réels :**
   - `MonthlyIncome` (Skewness = 1.37) : la médiane (4 919 $) est nettement inférieure à la moyenne (6 503 $). 80% des collaborateurs gagnent moins de 8 000 $, et une minorité de cadres dirigeants étire la distribution jusqu'à 20 000 $ (114 outliers légitimes).
   - `YearsAtCompany` (Skewness = 1.76) et `YearsSinceLastPromotion` (Skewness = 1.98) : très forte concentration sur les premières années d'ancienneté (médiane à 5 ans et 1 an respectivement).
   - *Implication ML :* Ces variables pourront bénéficier d'une transformation logarithmique ($\log(1+x)$) pour les modèles sensibles à la distribution (régression logistique).

2. **Variables symétriques ou proches de la normale :**
   - `Age` (Skewness = 0.41) suit une distribution équilibrée en cloche centrée sur 36 ans (min 18, max 60).

3. **Variables uniformes (artificielles) :**
   - `DailyRate`, `HourlyRate` et `MonthlyRate` présentent des distributions quasi plates et une asymétrie nulle (Skewness ≈ 0). Elles semblent générées uniformément et n'apportent que peu de structure empirique.


---

### 3.3. Analyse des Variables Catégorielles (Ordinales & Nominales)
Nous étudions la répartition des fréquences relatives au sein des catégories ordinales (évaluations hiérarchiques) et des catégories nominales (postes, départements, etc.).


In [ ]:
# Distribution des variables ordinales (notes de 1 à 4 ou 1 à 5)
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
total = len(df)

for i, col in enumerate(ordinal_columns):
    ax = axes[i]
    order = sorted(df[col].unique())
    sns.countplot(x=col, data=df, order=order, ax=ax, color="#2b5c8f")
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Effectif")
    
    for p in ax.patches:
        h = p.get_height()
        if h > 0:
            ax.annotate(f"{h/total*100:.0f}%", (p.get_x() + p.get_width() / 2., h + 8),
                        ha="center", fontsize=9)
    ax.set_ylim(0, max(df[col].value_counts()) * 1.15)

plt.tight_layout()
plt.show()


In [ ]:
# Distribution des variables nominales (triées par fréquence décroissante)
fig, axes = plt.subplots(4, 2, figsize=(14, 14))
axes = axes.flatten()

for i, col in enumerate(nominal_columns):
    ax = axes[i]
    order = df[col].value_counts().index
    sns.countplot(y=col, data=df, order=order, ax=ax, palette="Blues_r", hue=col, legend=False)
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_xlabel("Effectif")
    ax.set_ylabel("")
    
    for p in ax.patches:
        width = p.get_width()
        pct = 100 * width / total
        ax.annotate(f"{int(width)} ({pct:.1f}%)", (width + 8, p.get_y() + p.get_height() / 2.),
                    va="center", fontsize=9)
    ax.set_xlim(0, max(df[col].value_counts()) * 1.25)

# Masquer le 8ème subplot vide (7 variables nominales)
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


### 💡 Synthèse de l'Analyse des Variables Catégorielles

1. **Variables ordinales (Évaluations & Niveaux) :**
   - **Satisfaction globale :** Plus de 60% des collaborateurs se déclarent satisfaits (notes 3 et 4) sur leur environnement de travail, leur implication et leur poste.
   - **Niveaux de poste (`JobLevel`) :** Forte concentration sur les niveaux 1 (37%) et 2 (36%), tandis que les niveaux cadres dirigeants (4 et 5) ne représentent que 14% de l'effectif.
   - **Stock-options (`StockOptionLevel`) :** Près de 43% des collaborateurs ne possèdent aucune action (niveau 0).

2. **Variables nominales (Organisation & Démographie) :**
   - **Départements :** La R&D domine l'entreprise (**65%**), suivie des Ventes (**30%**) et des RH (**4%**).
   - **Postes (`JobRole`) :** Les rôles les plus fréquents sont *Sales Executive* (22%), *Research Scientist* (20%) et *Laboratory Technician* (18%).
   - **Heures supplémentaires (`OverTime`) :** 28% des collaborateurs effectuent des heures supplémentaires régulières.


## 4. Analyse Bivariée : Facteurs Associés à l'Attrition

L'objectif est d'identifier quelles caractéristiques différencient significativement les collaborateurs qui démissionnent (`Attrition = Yes`) de ceux qui restent (`Attrition = No`).  
Notre niveau de référence est la **moyenne globale d'attrition de l'entreprise : 16.1%**.

---

### 4.1. Variables Numériques vs Attrition


In [ ]:
# 1. Encodage binaire temporaire pour calculer les taux moyens
df['Attrition_num'] = (df['Attrition'] == 'Yes').astype(int)
baseline_attrition = df['Attrition_num'].mean() * 100

# 2. Tableau comparatif : moyenne, médiane et écart relatif
num_biv = []
for col in numeric_columns:
    val_yes = df[df['Attrition'] == 'Yes'][col]
    val_no = df[df['Attrition'] == 'No'][col]
    diff_pct = ((val_yes.mean() - val_no.mean()) / val_no.mean()) * 100
    
    num_biv.append({
        'Variable': col,
        'Moyenne (No)': round(val_no.mean(), 2),
        'Moyenne (Yes)': round(val_yes.mean(), 2),
        'Écart relatif (%)': f"{diff_pct:+.1f}%",
        'Médiane (No)': round(val_no.median(), 2),
        'Médiane (Yes)': round(val_yes.median(), 2)
    })

num_biv_df = pd.DataFrame(num_biv)
num_biv_df['abs_diff'] = num_biv_df['Écart relatif (%)'].str.rstrip('%').astype(float).abs()
num_biv_df = num_biv_df.sort_values(by='abs_diff', ascending=False).drop(columns=['abs_diff'])

display(num_biv_df)


In [ ]:
# Grille de boxplots comparatifs pour chaque variable numérique selon le statut d'Attrition
fig, axes = plt.subplots(5, 3, figsize=(15, 16))
axes = axes.flatten()

palette = {'No': '#4C72B0', 'Yes': '#C44E52'}

for i, col in enumerate(numeric_columns):
    ax = axes[i]
    sns.boxplot(x='Attrition', y=col, data=df, ax=ax, palette=palette, hue='Attrition', legend=False)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

# Masquer le 15ème subplot vide
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


### 💡 Enseignements clés (Numériques vs Attrition)

1. **Rôle déterminant de l'ancienneté et de l'expérience :**
   - Les collaborateurs qui partent ont **~35% moins d'ancienneté dans leur rôle** (`YearsInCurrentRole` : 2.9 ans vs 4.5 ans) et avec leur manager (`YearsWithCurrManager` : 2.8 ans vs 4.4 ans).
   - L'expérience globale accumulée (`TotalWorkingYears`) est de 8.2 ans chez les partants contre 11.9 ans chez les restants (-30%).

2. **Impact du niveau de salaire (`MonthlyIncome`) :**
   - Le salaire médian des collaborateurs quittant l'entreprise est de **3 202 $**, contre **5 204 $** pour ceux qui restent (un écart médian de **2 002 $**, soit -38% en médiane !). Les bas salaires présentent une vulnérabilité très nette.
   - *Précision :* Ne pas confondre `MonthlyIncome` (salaire réel médian à 3 202 $ / 5 204 $) avec `MonthlyRate` (taux facturé théorique avec une médiane proche de 14 500 $ sans écart entre les groupes).

3. **Impact de l'éloignement (`DistanceFromHome`) :**
   - Les partants résident en moyenne **20% plus loin** de leur lieu de travail (10.6 km vs 8.9 km).

4. **Variables neutres sans pouvoir discriminant :**
   - `HourlyRate` (65.6 $ vs 66.0 $), `MonthlyRate` (médiane ~14 600 $ vs ~14 120 $) et `PercentSalaryHike` (15.1% vs 15.2%) ne montrent aucun écart sensible entre les deux groupes.


---

### 4.2. Variables Catégorielles Ordinales vs Attrition
Nous calculons le taux effectif d'attrition pour chaque note/palier, comparé à la moyenne globale (ligne rouge à **16.1%**).


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
axes = axes.flatten()

for i, col in enumerate(ordinal_columns):
    ax = axes[i]
    rate_df = df.groupby(col)['Attrition_num'].agg(['mean', 'count']).reset_index()
    rate_df['rate_pct'] = rate_df['mean'] * 100
    
    sns.barplot(x=col, y='rate_pct', data=rate_df, ax=ax, color='#2b5c8f')
    
    # Ligne de référence globale (16.1%)
    ax.axhline(baseline_attrition, color='red', linestyle='--', linewidth=1.3, label=f'Moyenne ({baseline_attrition:.1f}%)')
    
    for p in ax.patches:
        h = p.get_height()
        ax.annotate(f'{h:.1f}%', (p.get_x() + p.get_width() / 2., h + 0.8),
                    ha='center', fontsize=9, fontweight='bold')
                    
    ax.set_title(f'Taux par {col}', fontsize=10, fontweight='bold')
    ax.set_ylabel("Taux d'attrition (%)")
    ax.set_xlabel('')
    ax.set_ylim(0, max(rate_df['rate_pct'].max(), baseline_attrition) * 1.25)
    if i == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


### 💡 Enseignements clés (Ordinales vs Attrition)

1. **`JobLevel` (Niveau hiérarchique) :**  
   Les employés de niveau 1 enregistrent un taux d'attrition record de **26.3%** (contre moins de 10% pour les niveaux 4 et 5). Les postes débutants/juniors concentrent le risque.
2. **`StockOptionLevel` (Options sur actions) :**  
   L'absence d'actions (niveau 0) est associée à **24.4%** de départ. L'octroi d'un niveau 1 ou 2 fait chuter le taux sous les **10%** : c'est un levier de fidélisation majeur.
3. **`EnvironmentSatisfaction` & `JobSatisfaction` :**  
   Le taux d'attrition diminue de façon quasi continue avec la satisfaction (de ~23% pour une note de 1 à ~11-13% pour une note de 4).
4. **Nuance sur `RelationshipSatisfaction` (Relations avec les collègues) :**  
   Le sur-risque de départ intervient **uniquement si la relation est conflictuelle / dégradée (note 1 à 20.7%)**. Dès que la relation atteint un niveau acceptable (notes 2, 3 ou 4), le taux se stabilise autour de 15% sans gain protecteur supplémentaire.


---

### 4.3. Variables Catégorielles Nominales vs Attrition


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 15))
axes = axes.flatten()

for i, col in enumerate(nominal_columns):
    ax = axes[i]
    rate_df = df.groupby(col)['Attrition_num'].agg(['mean', 'count']).reset_index()
    rate_df['rate_pct'] = rate_df['mean'] * 100
    rate_df = rate_df.sort_values(by='rate_pct', ascending=False)
    
    sns.barplot(y=col, x='rate_pct', data=rate_df, ax=ax, palette='Reds_r', hue=col, legend=False)
    ax.axvline(baseline_attrition, color='black', linestyle='--', linewidth=1.3, label=f'Moyenne ({baseline_attrition:.1f}%)')
    
    for p in ax.patches:
        w = p.get_width()
        ax.annotate(f'{w:.1f}%', (w + 0.8, p.get_y() + p.get_height() / 2.),
                    va='center', fontsize=9, fontweight='bold')
                    
    ax.set_title(f'Taux par {col}', fontsize=10, fontweight='bold')
    ax.set_xlabel("Taux d'attrition (%)")
    ax.set_ylabel('')
    ax.set_xlim(0, max(rate_df['rate_pct'].max(), baseline_attrition) * 1.25)
    if i == 0:
        ax.legend(fontsize=8)

# Masquer le 8ème subplot vide (7 nominales)
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()


### 💡 Enseignements clés (Nominales vs Attrition)

1. **`OverTime` (Heures supplémentaires) — Variable présentant l'une des plus fortes différences de taux d'attrition observées dans l'analyse bivariée. :**  
   Les employés astreints à des heures supplémentaires ont un taux de départ de **30.5%** contre seulement **10.4%** pour ceux qui n'en font pas (un risque multiplié par 3 !).
2. **`JobRole` (Disparités sectorielles fortes) :**  
   - Postes à très fort turnover : *Sales Representative* (**39.8%**) et *Laboratory Technician* (**23.9%**).
   - Postes à très forte rétention : *Research Director* (**2.5%**) et *Manager* (**4.9%**).
3. **`MaritalStatus` (Situation familiale) :**  
   Les célibataires démissionnent deux fois et demie plus (**25.5%**) que les personnes mariées (**12.5%**) ou divorcées (**10.1%**), reflétant une plus grande mobilité.
4. **`BusinessTravel` (Fréquence des voyages) :**  
   Les déplacements fréquents augmentent significativement l'attrition (**24.9%** vs 8.0% pour les non-voyageurs).


## 5. Analyse Multivariée & Multicolinéarité

L'analyse multivariée vise à étudier les relations simultanées entre les différentes variables explicatives afin d'identifier les **phénomènes de multicolinéarité** (redondances d'information) qui peuvent déstabiliser certains algorithmes de modélisation (ex: régression logistique).


In [ ]:
# Sélection des variables continues et ordinales clés pour l'étude des corrélations
corr_cols = [
    'Age', 'MonthlyIncome', 'JobLevel', 'TotalWorkingYears',
    'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
    'YearsWithCurrManager', 'PercentSalaryHike', 'PerformanceRating',
    'DistanceFromHome', 'NumCompaniesWorked', 'TrainingTimesLastYear',
    'StockOptionLevel'
]

corr_matrix = df[corr_cols].corr()

# Masque pour cacher le triangle supérieur redondant
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Matrice de Corrélation Linéaire des Variables Clés (Pearson)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### 💡 Enseignements Clés sur la Multicolinéarité

1. **Quasi-redondance entre Salaire et Niveau Hiérarchique :**  
   - `MonthlyIncome` et `JobLevel` présentent une forte corrélation de **0.95**, suggérant une forte redondance informationnelle potentielle pour certains modèles.
2. **Bloc d'ancienneté hautement colinéaire :**  
   - `YearsAtCompany`, `YearsInCurrentRole` et `YearsWithCurrManager` sont fortement corrélées entre elles (**0.76 à 0.77**).
   - `TotalWorkingYears` est également fortement liée à `JobLevel` (**0.78**) et `MonthlyIncome` (**0.77**).
3. **Lien Performance / Augmentation :**  
   - `PercentSalaryHike` et `PerformanceRating` sont fortement associés dans cet échantillon.


## 6. Synthèse Générale & Recommandations pour la Modélisation

### 📌 Portrait-robot du collaborateur à haut risque d'attrition
L'analyse exploratoire met en lumière une combinaison de facteurs de risque :
- **Condition de travail :** Effectuer des heures supplémentaires régulières (`OverTime = Yes`).
- **Niveau d'entrée & Rémunération :** Poste de début de carrière (`JobLevel = 1`, salaire médian bas ≈ 3 200 $).
- **Rôles spécifiques :** Commerciaux de terrain (*Sales Representative*, 40% de départ) et techniciens de laboratoire (*Lab Tech*, 24%).
- **Intéressement :** Absence d'options sur actions (`StockOptionLevel = 0`).
- **Profil personnel :** Jeune âge, célibataire, et résidant loin de l'entreprise (`DistanceFromHome` élevé).

**Il ne faut surtout pas conclure qu'une personne remplissant tous ces critères quittera l'entreprise, mais ces signaux combinés augmentent fortement le risque de départ.**